# ML Classification: Gesture Phase Segmentation
**MSc Data Science — Machine Learning Group Submission**

**Dataset:** OpenML ID 4538 — 9,873 samples, 32 features, 5 classes (D, H, P, R, S)  
**Split:** 70/30 stratified | **Scoring:** balanced_accuracy  
**Models:** SVM (RBF), Random Forest, KNN, LightGBM, Extra Trees, MLP, LDA, Naive Bayes, Logistic Regression

In [ ]:
import subprocess, sys
for pkg, imp in [('lightgbm', 'lightgbm'), ('scikit-learn', 'sklearn'), ('seaborn', 'seaborn')]:
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages available.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
dataset = fetch_openml(data_id=4538, as_frame=False)
X, y = dataset.data, dataset.target
CLASSES = np.unique(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
y_test_bin = label_binarize(y_test, classes=CLASSES)

print(f'Samples: {X.shape[0]}  Features: {X.shape[1]}  Classes: {list(CLASSES)}')
print(f'Train: {len(X_train)}  Test: {len(X_test)}')
classes_u, counts = np.unique(y, return_counts=True)
print(pd.DataFrame({'Class': classes_u, 'Count': counts, '%': (counts/len(y)*100).round(1)}).to_string(index=False))

In [ ]:
all_results, all_roc_data = {}, {}

def show_cv_results(search, top_n=10):
    cv_df = pd.DataFrame(search.cv_results_)
    param_cols = [c for c in cv_df.columns if c.startswith('param_')]
    cv_df = cv_df[param_cols + ['mean_test_score', 'rank_test_score']].sort_values('rank_test_score').head(top_n)
    cv_df.columns = [c.replace('param_', '') for c in cv_df.columns]
    print(cv_df.round(4).to_string(index=False))

def evaluate(name, y_true, y_pred, y_prob):
    bal_acc   = balanced_accuracy_score(y_true, y_pred)
    macro_auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
    micro_auc = roc_auc_score(y_test_bin, y_prob, average='micro')

    print(f'\n=== {name} ===')
    print(f'Balanced Accuracy: {bal_acc:.4f}  Macro AUC: {macro_auc:.4f}  Micro AUC: {micro_auc:.4f}')
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred, labels=CLASSES), display_labels=CLASSES
    ).plot(cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout(); plt.show()

    fpr, tpr, roc_auc_vals = {}, {}, {}
    for i, cls in enumerate(CLASSES):
        fpr[cls], tpr[cls], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc_vals[cls] = auc(fpr[cls], tpr[cls])

    fig, ax = plt.subplots(figsize=(7, 5))
    for cls in CLASSES:
        ax.plot(fpr[cls], tpr[cls], label=f'{cls} (AUC={roc_auc_vals[cls]:.3f})')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'Per-Class OvR ROC - {name}')
    ax.legend(loc='lower right')
    plt.tight_layout(); plt.show()

    all_results[name] = {'balanced_accuracy': bal_acc, 'macro_auc': macro_auc, 'micro_auc': micro_auc}
    all_roc_data[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc_vals}

## Model 1: Support Vector Machine (RBF Kernel)
SVM with RBF kernel finds an optimal separating hyperplane in high-dimensional space. `StandardScaler` is applied inside a `Pipeline`. `RandomizedSearchCV` explores C and gamma.

In [ ]:
svm_search = RandomizedSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))]),
    {'clf__C': [0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
     'clf__gamma': ['scale', 'auto', 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]},
    n_iter=25, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
svm_search.fit(X_train, y_train)
print(f'Best params: {svm_search.best_params_}  CV score: {svm_search.best_score_:.4f}')
show_cv_results(svm_search)
evaluate('SVM (RBF)', y_test,
         svm_search.best_estimator_.predict(X_test),
         svm_search.best_estimator_.predict_proba(X_test))

## Model 2: Random Forest
Ensemble of decision trees trained on bootstrapped subsets with random feature selection. No scaling required. `RandomizedSearchCV` over tree structure hyperparameters.

In [ ]:
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {'n_estimators': [50, 100, 200, 300, 500], 'max_depth': [None, 5, 10, 20, 30],
     'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 4, 8],
     'max_features': ['sqrt', 'log2', 0.3, 0.5]},
    n_iter=40, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
rf_search.fit(X_train, y_train)
print(f'Best params: {rf_search.best_params_}  CV score: {rf_search.best_score_:.4f}')
show_cv_results(rf_search)
evaluate('Random Forest', y_test,
         rf_search.best_estimator_.predict(X_test),
         rf_search.best_estimator_.predict_proba(X_test))

## Model 3: K-Nearest Neighbours
Instance-based learner classifying by majority vote among k nearest neighbours. Scale-sensitive — `StandardScaler` inside `Pipeline`. `GridSearchCV` over k, weights, and distance metric.

In [ ]:
knn_search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_jobs=-1))]),
    {'clf__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
     'clf__weights': ['uniform', 'distance'],
     'clf__metric': ['euclidean', 'manhattan', 'minkowski']},
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, verbose=1
)
knn_search.fit(X_train, y_train)
print(f'Best params: {knn_search.best_params_}  CV score: {knn_search.best_score_:.4f}')
show_cv_results(knn_search)
evaluate('KNN', y_test,
         knn_search.best_estimator_.predict(X_test),
         knn_search.best_estimator_.predict_proba(X_test))

## Model 4: LightGBM
Fast gradient boosting with leaf-wise tree growth and histogram-based splits. No scaling required. `RandomizedSearchCV` over boosting hyperparameters.

In [ ]:
lgbm_search = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
    {'n_estimators': [50, 100, 200, 300, 500], 'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
     'num_leaves': [15, 31, 63, 127], 'max_depth': [-1, 3, 5, 7, 10],
     'min_child_samples': [10, 20, 30, 50],
     'subsample': [0.6, 0.7, 0.8, 0.9, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]},
    n_iter=40, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
lgbm_search.fit(X_train, y_train)
print(f'Best params: {lgbm_search.best_params_}  CV score: {lgbm_search.best_score_:.4f}')
show_cv_results(lgbm_search)
evaluate('LightGBM', y_test,
         lgbm_search.best_estimator_.predict(X_test),
         lgbm_search.best_estimator_.predict_proba(X_test))

## Model 5: Extra Trees
Like Random Forest but with fully randomised split thresholds. Lower variance, faster training. No scaling required. Same hyperparameter space as Random Forest.

In [ ]:
et_search = RandomizedSearchCV(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {'n_estimators': [50, 100, 200, 300, 500], 'max_depth': [None, 5, 10, 20, 30],
     'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 4, 8],
     'max_features': ['sqrt', 'log2', 0.3, 0.5]},
    n_iter=40, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
et_search.fit(X_train, y_train)
print(f'Best params: {et_search.best_params_}  CV score: {et_search.best_score_:.4f}')
show_cv_results(et_search)
evaluate('Extra Trees', y_test,
         et_search.best_estimator_.predict(X_test),
         et_search.best_estimator_.predict_proba(X_test))

## Model 6: Multi-Layer Perceptron
Feedforward neural network trained with backpropagation. Scale-sensitive — `StandardScaler` inside `Pipeline`. `RandomizedSearchCV` over architecture and regularisation.

In [ ]:
mlp_search = RandomizedSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', MLPClassifier(max_iter=500, random_state=RANDOM_STATE))]),
    {'clf__hidden_layer_sizes': [(64,), (128,), (256,), (64, 64), (128, 64),
                                  (128, 128), (256, 128), (64, 64, 32)],
     'clf__activation': ['relu', 'tanh'],
     'clf__alpha': [1e-5, 1e-4, 1e-3, 0.01, 0.1],
     'clf__learning_rate': ['constant', 'adaptive']},
    n_iter=30, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
mlp_search.fit(X_train, y_train)
print(f'Best params: {mlp_search.best_params_}  CV score: {mlp_search.best_score_:.4f}')
show_cv_results(mlp_search)
evaluate('MLP', y_test,
         mlp_search.best_estimator_.predict(X_test),
         mlp_search.best_estimator_.predict_proba(X_test))

## Model 7: Linear Discriminant Analysis
Generative model that projects features onto directions maximising class separability. `GridSearchCV` handles the solver–shrinkage interaction via a list of parameter grids.

In [ ]:
lda_search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', LinearDiscriminantAnalysis())]),
    [
        {'clf__solver': ['svd'], 'clf__n_components': [1, 2, 3, 4]},
        {'clf__solver': ['lsqr', 'eigen'],
         'clf__shrinkage': [None, 'auto', 0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
         'clf__n_components': [1, 2, 3, 4]}
    ],
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, verbose=1
)
lda_search.fit(X_train, y_train)
print(f'Best params: {lda_search.best_params_}  CV score: {lda_search.best_score_:.4f}')
show_cv_results(lda_search)
evaluate('LDA', y_test,
         lda_search.best_estimator_.predict(X_test),
         lda_search.best_estimator_.predict_proba(X_test))

## Model 8: Naive Bayes (Gaussian)
Applies Bayes' theorem assuming conditional feature independence. `GridSearchCV` over `var_smoothing` to stabilise covariance estimates.

In [ ]:
nb_search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]),
    {'clf__var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]},
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, verbose=1
)
nb_search.fit(X_train, y_train)
print(f'Best params: {nb_search.best_params_}  CV score: {nb_search.best_score_:.4f}')
show_cv_results(nb_search, top_n=11)
evaluate('Naive Bayes', y_test,
         nb_search.best_estimator_.predict(X_test),
         nb_search.best_estimator_.predict_proba(X_test))

## Model 9: Logistic Regression
Linear classifier modelling log-odds with multinomial softmax. `GridSearchCV` handles the solver–penalty interaction via a list of parameter grids. Scaling inside `Pipeline`.

In [ ]:
lr_search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    [
        {'clf__C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
         'clf__solver': ['lbfgs'], 'clf__penalty': ['l2']},
        {'clf__C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
         'clf__solver': ['saga'], 'clf__penalty': ['l1', 'l2']}
    ],
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy', n_jobs=-1, verbose=1
)
lr_search.fit(X_train, y_train)
print(f'Best params: {lr_search.best_params_}  CV score: {lr_search.best_score_:.4f}')
show_cv_results(lr_search)
evaluate('Logistic Regression', y_test,
         lr_search.best_estimator_.predict(X_test),
         lr_search.best_estimator_.predict_proba(X_test))

## Final Comparison

In [ ]:
summary = pd.DataFrame(all_results).T
summary.columns = ['Bal. Acc', 'Macro AUC', 'Micro AUC']
summary = summary.sort_values('Macro AUC', ascending=False)
print(summary.round(4).to_string())

colors = plt.cm.tab10(np.linspace(0, 0.9, len(summary)))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['Bal. Acc', 'Macro AUC', 'Micro AUC']):
    ax.barh(summary.index[::-1], summary[col][::-1], color=colors)
    ax.set_title(col); ax.grid(axis='x', alpha=0.4)
plt.suptitle('Model Comparison - Gesture Phase Segmentation', fontweight='bold')
plt.tight_layout(); plt.show()

plot_colors = plt.cm.tab10(np.linspace(0, 0.9, len(all_roc_data)))
for cls in CLASSES:
    fig, ax = plt.subplots(figsize=(8, 5))
    for (name, data), color in zip(all_roc_data.items(), plot_colors):
        ax.plot(data['fpr'][cls], data['tpr'][cls], color=color,
                label=f'{name} (AUC={data["auc"][cls]:.3f})')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC Comparison - Class "{cls}"')
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout(); plt.show()